In [ ]:
# Importing the python libraries
import pandas as pd  # for Data preprocessing
import matplotlib.pyplot as plt  # for Data Visualization
import seaborn as sns  # for Data Visualization
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix,precision_score,recall_score,f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score


In [ ]:
cleaned_customer_transaction_data = pd.read_csv(r'C:\Users\oyebi\Fraudulent-Transaction-Detection\Dataset\artifacts\EDA_data.csv')

### Threshold-Based Feature Engineering

Threshold-based features were created based on key behavioural and risk indicators identified during exploratory data analysis. These included **low KYC verification levels**, **location mismatches between the registered country and IP address**, and high-risk transaction characteristics such as **high transaction amounts**, **elevated IP risk scores**, and **low device trust scores**.

Specific thresholds were identified based on categories that exhibited significantly higher fraud rates during the exploratory analysis. These thresholds were then converted into **binary risk indicators** to enhance the model’s ability to identify potentially suspicious transactions.

In [ ]:
# Creating threshold-based features from the following risk signals

cleaned_customer_transaction_data['timestamp'] = pd.to_datetime(cleaned_customer_transaction_data['timestamp']) # Ensure timestamp is in datetime format
cleaned_customer_transaction_data['late_night_hours'] = ((cleaned_customer_transaction_data['hour'] >= 3) & (cleaned_customer_transaction_data['hour'] <= 7)).astype(int) # Transactions occurring between 03:00 and 07:00
cleaned_customer_transaction_data['amount_high'] = (cleaned_customer_transaction_data['amount_usd'] >= 1000).astype(int) # High-value transaction
cleaned_customer_transaction_data['high_ip_risk'] = (cleaned_customer_transaction_data['ip_risk_score'] > 0.8).astype(int) # High IP risk
cleaned_customer_transaction_data['low_device_trust'] = (cleaned_customer_transaction_data['device_trust_score'] < 0.5).astype(int)
cleaned_customer_transaction_data['new_account'] = ((cleaned_customer_transaction_data['account_age_days'] >= 30) & (cleaned_customer_transaction_data['account_age_days'] < 90)).astype(int) # New account: 30 to less than 90 days old
cleaned_customer_transaction_data['very_new_account'] = (cleaned_customer_transaction_data['account_age_days'] <30).astype(int) # Very new account: less than 30 days old
cleaned_customer_transaction_data['velocity_spike'] = (cleaned_customer_transaction_data['txn_velocity_1h'] >= 3).astype(int)


high_risk_signal_features = [
    'late_night_hours',
    'amount_high',
    'high_ip_risk',
    'low_device_trust',
    'new_account',
    'very_new_account',
    'velocity_spike'
]



In [ ]:
cleaned_customer_transaction_data[high_risk_signal_features].head()

,late_night_hours,amount_high,high_ip_risk,low_device_trust,new_account,very_new_account,velocity_spike
0,0,0,0,0,0,0,0
1,0,0,0,1,0,0,0
2,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0


In [ ]:
# Comparing each risk signal with the fraud target:

for feature in high_risk_signal_features:
    print(f"\nFraud rate by {feature}:")
    print(
        cleaned_customer_transaction_data
        .groupby(feature)['is_fraud']
        .agg(['count', 'sum', 'mean'])
    )



Fraud rate by late_night_hours:
                  count  sum      mean
late_night_hours                      
0                  8423  570  0.067672
1                  2357  411  0.174374

Fraud rate by amount_high:
             count  sum      mean
amount_high                      
0            10208  688  0.067398
1              572  293  0.512238

Fraud rate by high_ip_risk:
              count  sum      mean
high_ip_risk                      
0              9543  207  0.021691
1              1237  774  0.625707

Fraud rate by low_device_trust:
                  count  sum      mean
low_device_trust                      
0                  8282  172  0.020768
1                  2498  809  0.323859

Fraud rate by new_account:
             count  sum      mean
new_account                      
0            10443  790  0.075649
1              337  191  0.566766

Fraud rate by very_new_account:
                  count  sum      mean
very_new_account                      
0             

### Evaluation of Threshold-Based Risk Features

Seven threshold-based binary features were created from risk patterns identified during exploratory data analysis. Each feature was evaluated by comparing the fraud rate when the risk condition was absent (`0`) with the fraud rate when the condition was present (`1`).

The analysis showed that all seven engineered features were associated with higher fraud rates when the corresponding risk condition was present.

- **Late-night hours:** Transactions occurring between 03:00 and 07:00 recorded a fraud rate of **17.44%**, compared with **6.77%** outside this period.
- **High transaction amount:** Transactions of at least **$1,000** recorded a fraud rate of **51.22%**, compared with **6.74%** for transactions below this threshold.
- **High IP risk:** Transactions with an `ip_risk_score > 0.8` recorded a fraud rate of **62.57%**, compared with only **2.17%** when this condition was absent.
- **Low device trust:** Transactions with a `device_trust_score < 0.5` recorded a fraud rate of **32.39%**, compared with **2.08%** when the condition was absent.
- **New account:** Transactions involving accounts aged **30 to less than 90 days** recorded a fraud rate of **56.68%**, compared with **7.56%** for transactions outside this age range.
- **Very new account:** Transactions involving accounts aged **less than 30 days** recorded a fraud rate of **36.66%**, compared with **3.52%** among older accounts.
- **Transaction velocity spike:** Transactions with at least **3 transactions within one hour** recorded the highest fraud rate of **76.65%**, compared with only **1.72%** when the velocity-spike condition was absent.

Overall, the threshold-based features demonstrated substantial differences in fraud occurrence between transactions with and without the identified risk signals. The strongest observed risk signal was **transaction velocity spike**, followed by **high IP risk**, **new account status**, and **high transaction amount**.

These findings suggest that the engineered features capture meaningful behavioural, transactional, device, geographic, and account-related risk patterns that may provide useful predictive information for the fraud detection model. However, these relationships represent associations rather than causal effects, and the predictive contribution of each engineered feature should be evaluated during model training and validation.


In [ ]:
# Checking the features in our dataset

list(cleaned_customer_transaction_data.columns)



['transaction_id',
 'customer_id',
 'timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'exchange_rate_src_to_dest',
 'device_id',
 'new_device',
 'ip_address',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'chargeback_history_count',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud',
 'hour',
 'day_of_week',
 'month',
 'is_weekend',
 'account_age_bucket',
 'device_trust_score_bucket',
 'ip_risk_score_bucket',
 'amount_usd_bucket',
 'late_night_hours',
 'amount_high',
 'high_ip_risk',
 'low_device_trust',
 'new_account',
 'very_new_account',
 'velocity_spike']

## Feature selection

### Deleting /removing non-model features

In [ ]:
# Dropping all temporary bucket columns
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['account_age_bucket', 'device_trust_score_bucket', 'ip_risk_score_bucket', 'amount_usd_bucket', ], axis=1)

# All identifiers columns
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['transaction_id', 'customer_id', 'device_id', 'ip_address'], axis=1)  # axis=1 means work on the columns.

# Dropping irrelevant columns/variables
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['chargeback_history_count', 'month','exchange_rate_src_to_dest'], axis=1)


### Defining Categorical Features

In [ ]:
# Identifying categorical features in the dataset
categorical_features = cleaned_customer_transaction_data.select_dtypes(include =['object', 'bool']).columns

categorical_features

Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'new_device', 'ip_country', 'location_mismatch', 'kyc_tier'],
      dtype='object')

In [ ]:
# Identifying numerical features in the dataset
numerical_features = cleaned_customer_transaction_data.select_dtypes(include =['int', 'float']).columns.drop('is_fraud')  # Exclude the target variable


numerical_features

Index(['amount_src', 'amount_usd', 'fee', 'ip_risk_score', 'account_age_days',
       'device_trust_score', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'hour', 'day_of_week',
       'is_weekend', 'late_night_hours', 'amount_high', 'high_ip_risk',
       'low_device_trust', 'new_account', 'very_new_account',
       'velocity_spike'],
      dtype='object')

In [ ]:
# To check the number (length) of features in each feature group

# Check the number of categorical features
print(f"Categorical features: {len(categorical_features)}")

# Check the number of numerical features
print(f"Numerical features: {len(numerical_features)}")

#checking the number of features in the dataset
print(f"Dataset: {cleaned_customer_transaction_data.shape}")  # shape[1] gives the number of columns/features in the dataset}

Categorical features: 8
Numerical features: 20
Dataset: (10780, 30)


In [ ]:
# checking the columns
cleaned_customer_transaction_data.columns



Index(['timestamp', 'home_country', 'source_currency', 'dest_currency',
       'channel', 'amount_src', 'amount_usd', 'fee', 'new_device',
       'ip_country', 'location_mismatch', 'ip_risk_score', 'kyc_tier',
       'account_age_days', 'device_trust_score', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'hour', 'day_of_week', 'is_weekend', 'late_night_hours', 'amount_high',
       'high_ip_risk', 'low_device_trust', 'new_account', 'very_new_account',
       'velocity_spike'],
      dtype='object')

In [ ]:
cleaned_customer_transaction_data.info()  # Display the summary of the DataFrame, including data types and non-null counts


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10780 entries, 0 to 10779
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   timestamp            10780 non-null  datetime64[ns, UTC]
 1   home_country         10756 non-null  object             
 2   source_currency      10780 non-null  object             
 3   dest_currency        10780 non-null  object             
 4   channel              10780 non-null  object             
 5   amount_src           10780 non-null  float64            
 6   amount_usd           10780 non-null  float64            
 7   fee                  10780 non-null  float64            
 8   new_device           10780 non-null  bool               
 9   ip_country           10752 non-null  object             
 10  location_mismatch    10780 non-null  bool               
 11  ip_risk_score        10780 non-null  float64            
 12  kyc_tier          

# Save the dataset with the newly created features to a new CSV file

In [ ]:
# Save the dataset with the newly created features to a new CSV file

cleaned_customer_transaction_data.to_csv("../Dataset/artifacts/Engineered_data.csv", index=False)
